[![gammapy](https://img.shields.io/badge/powered%20by-gammapy-orange.svg?style=flat)](https://gammapy.org/)
# CTAO Visibility Analysis


This notebook demonstrates how to compute the annual visibility of a sky target
for CTAO observatories using the CTAOVisibilityEstimator.


We include realistic observational constraints:
- Astronomical night (Sun altitude)
- Moon avoidance
- Optional airmass weighting


The results can be used to:
- Plan observations
- Estimate exposure
- Compare CTAO North vs South performance

<a id='intro'></a>
## Index

* [**1. Introduction**](#intro_sec)

* [**2. Setup**](#setup)

* [**3. Define Target**](#target)
  * We define the sky position using:
    - `astropy.coordinates.SkyCoord`

* [**4. Compute Visibility**](#compute)
  * We compute visibility using:
    - `CTAOVisibilityEstimator`
    - Astronomical constraints (Sun, Moon)

* [**5. Compare CTAO Sites**](#compare)
  * Comparison between:
    - CTAO South
    - CTAO North

* [**6. Airmass Weighting**](#airmass)
  * Optional weighting using:
    - Airmass approximation (`1 / cos(z)`)

* [**7. Create Tables**](#table)
  * Generate summary outputs with:
    - `pandas.DataFrame`

* [**8. Interpretation**](#interpretation)
  * Scientific interpretation of visibility results

In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u

from feupy.irf import (
CTAOVisibilityEstimator,
make_ctao_visibility_table,
)

import matplotlib.pyplot as plt

## Define Target Position
We define a sky position using `SkyCoord`.
In this example, we use the position of Centaurus A, a nearby radio galaxy commonly used in CTAO studies.

In [22]:
target = SkyCoord(
ra=201.3651 * u.deg,
dec=-43.0191 * u.deg,
frame="icrs"
)

target

<SkyCoord (ICRS): (ra, dec) in deg
    (201.3651, -43.0191)>

In [ ]:





# In[2]:



# ## Initialize Visibility Estimator

# In[3]:

estimator = CTAOVisibilityEstimator(
target=target,
year=2025,
time_step_min=30,
)

print(estimator)

# ## Compute Visibility (CTAO South)

# In[4]:

vis_south = estimator.compute_visibility("cta_south")
vis_south

# ## Compare CTAO North and South

# In[5]:

vis_north = estimator.compute_visibility("cta_north")

print("South:", vis_south)
print("North:", vis_north)

# ## Plot Visibility Comparison

# In[6]:

labels = list(vis_south.keys())
south_vals = list(vis_south.values())
north_vals = list(vis_north.values())

plt.figure()
plt.plot(labels, south_vals, marker="o", label="South")
plt.plot(labels, north_vals, marker="o", label="North")

plt.xlabel("Zenith bin (deg)")
plt.ylabel("Visibility (hours)")
plt.title("CTAO Visibility Comparison")
plt.legend()
plt.show()

# ## Airmass-Weighted Visibility

#

# Visibility can optionally be weighted by airmass,

# giving more importance to low-zenith observations.

# In[7]:

estimator_weighted = CTAOVisibilityEstimator(
target=target,
year=2025,
time_step_min=30,
use_airmass_weight=True,
)

vis_weighted = estimator_weighted.compute_visibility("cta_south")
vis_weighted

# ## Create Visibility Table

# In[8]:

df = make_ctao_visibility_table(estimator)
df

# ## Save Results

# In[9]:

make_ctao_visibility_table(estimator, save_path="ctao_visibility.csv")

# ## Interpretation

#

# - Visibility is given in hours per year

# - Each bin corresponds to a zenith angle range:

# - 20° → low zenith (best sensitivity, lowest energy threshold)

# - 60° → high zenith (reduced sensitivity, higher energy threshold)

#

# Scientific implications:

# - Sources with high visibility at low zenith are ideal for deep observations

# - Visibility impacts achievable sensitivity and exposure

# - Comparing North vs South helps determine optimal CTAO site

#

# These results can be used to:

# - Optimize observation strategies

# - Weight simulations

# - Estimate realistic exposure

# ## Next Steps

#

# - Combine visibility with IRFs for realistic simulations

# - Integrate into CTAOAnalysis workflows

# - Perform sensitivity studies


In [1]:
from astropy.coordinates import SkyCoord
import astropy.units as u

from feupy.irf import (
    CTAOIRFManager,
    CTAOVisibilityEstimator,
    make_ctao_visibility_table,
)

## Define Target Position

We define a sky position using `SkyCoord`.
Here we use a Crab-like position.

In [6]:
# Example: Centaurus A
target = SkyCoord(
    ra=201.3651 * u.deg,
    dec=-43.0191 * u.deg,
    frame="icrs"
)

## Compute Annual Visibility

We compute the annual visibility for a given target position.

Constraints included:
- Astronomical night (Sun altitude)
- Moon avoidance

In [18]:
estimator = CTAOVisibilityEstimator(
    target=target,
    year=2025,
    time_step_min=30,
)

visibility_south = estimator.compute_visibility("cta_south")
visibility_south

cta_south: 100%|██████████████████████████████| 365/365 [00:16<00:00, 22.46it/s]


{'20': np.float64(345.5), '40': np.float64(336.5), '60': np.float64(319.0)}

## Compare CTAO North and South

In [16]:
vis_south = estimator.compute_visibility("cta_south")
vis_north = estimator.compute_visibility("cta_north")

print("South:", vis_south)
print("North:", vis_north)

cta_north: 100%|██████████████████████████████| 365/365 [00:17<00:00, 21.09it/s]

South: {'20': <Quantity 314.54749576>, '40': <Quantity 253.81603101>, '60': <Quantity 155.44372327>}
North: {'20': <Quantity 0.>, '40': <Quantity 0.>, '60': <Quantity 0.>}


## Airmass-Weighted Visibility

Optionally, visibility can be weighted by airmass.

In [17]:
estimator_weighted = CTAOVisibilityEstimator(
    target=target,
    year=2025,
    time_step_min=30,
    use_airmass_weight=True,
)

vis_weighted = estimator_weighted.compute_visibility("cta_south")

vis_weighted

cta_south: 100%|██████████████████████████████| 365/365 [00:16<00:00, 21.51it/s]


{'20': <Quantity 318.27553421>,
 '40': <Quantity 257.39943693>,
 '60': <Quantity 158.42077593>}

## Create Visibility Table

We generate a summary table for both CTAO sites.

In [20]:
df = make_ctao_visibility_table(estimator)

df

cta_north: 100%|██████████████████████████████| 365/365 [00:17<00:00, 20.66it/s]


,Observatory,Zenith (deg),Visibility (hours)
3,CTAO North,20,0.0
4,CTAO North,40,0.0
5,CTAO North,60,0.0
0,CTAO South,20,345.5
1,CTAO South,40,336.5
2,CTAO South,60,319.0


## Save Results

The table can be saved to CSV or LaTeX format.

In [21]:
make_ctao_visibility_table(estimator, save_path="visibility.csv")

cta_north: 100%|██████████████████████████████| 365/365 [00:17<00:00, 20.70it/s]


,Observatory,Zenith (deg),Visibility (hours)
3,CTAO North,20,0.0
4,CTAO North,40,0.0
5,CTAO North,60,0.0
0,CTAO South,20,345.5
1,CTAO South,40,336.5
2,CTAO South,60,319.0


## Interpretation

- The visibility is given in hours per year.
- Each bin corresponds to a zenith angle range:
  - 20° → low zenith (best performance)
  - 60° → higher zenith (worse sensitivity)

These results can be used to:
- Select optimal observation strategies
- Weight simulations
- Estimate realistic exposure

## Next Steps

- Combine visibility with IRFs for realistic simulations
- Integrate into CTAOAnalysis workflows
- Perform sensitivity studies